Notebook 6c — Joint Training: PTB-XL + INCART

Per the spec: train the model jointly on **PTB-XL** and **St. Petersburg
INCART** (both genuine 12-lead datasets already used elsewhere in this
pipeline), then hand off to **Notebook 6d** for cross-dataset validation
on **CPSC 2018** and **Chapman-Shaoxing** (held out — never seen during
training).

What this notebook does

✅ Loads PTB-XL splits already produced by NB1/NB2 (`train_processed.npz`,
`val_processed.npz`)

✅ Loads INCART directly from raw WFDB records (same loader logic as NB6),
extracts 10‑second 12‑lead segments, labels them via the AAMI beat
annotation → risk mapping, and splits them into an INCART‑train /
INCART‑val partition (record‑level split, so no leakage between splits)

✅ Concatenates PTB‑XL‑train + INCART‑train into one joint training set,
and PTB‑XL‑val + INCART‑val into one joint validation set for early
stopping

✅ Re‑uses the exact training recipe from NB4 (`ECGRiskNetXAI`,
`ClassBalancedFocalLoss` + `SupConLoss` + auxiliary‑head `FocalLoss`,
MixUp, cosine warmup, SWA, gradient clipping, early stopping on val
macro‑F1)

✅ Saves the new joint checkpoint as `joint_model.pt` (and SWA variant
`joint_swa_model.pt`) — **does not overwrite** `best_model.pt` from NB4,
so the original PTB‑XL‑only model stays intact for comparison

✅ Reports **separate** per‑source metrics on the held‑out validation
split (PTB‑XL‑val alone, INCART‑val alone, and the pooled joint‑val) so
you can see whether INCART hurts/helps PTB‑XL performance and vice versa

✅ Saves `joint_training_results.json` with all the numbers, plus
training curves and per‑source confusion matrices

Note: This notebook trains a **separate** checkpoint from NB4's
`best_model.pt`. Notebook 6d evaluates *this* joint checkpoint
(`joint_model.pt`) on CPSC 2018 and Chapman.

In [ ]:
import os, sys, json, time, warnings, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

import wfdb
from scipy.signal import resample as scipy_resample

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, average_precision_score,
    precision_score, recall_score, roc_curve, precision_recall_curve
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = Path("./ECG_Project/data")
sys.path.insert(0, str(SAVE_DIR))

from ecg_model import ECGRiskNetXAI, ClassBalancedFocalLoss, SupConLoss, FocalLoss

with open(SAVE_DIR / "metadata.json") as f:
    META = json.load(f)

RISK_LABELS = {int(k): v for k, v in META["risk_labels"].items()}
MITBIH_TO_RISK = META.get("mitbih_to_risk", {"N": 0, "S": 1, "V": 2, "F": 2, "Q": 3})
STANDARD_LEAD_ORDER = META.get("standard_lead_order",
    ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"])
TARGET_LEN = META.get("target_len", 1000)
FS_TARGET = META.get("fs_target", 100)

print("✅ Setup complete.")
print(f"Device: {device}")
print(f"Risk labels: {RISK_LABELS}")

Step 1: Load PTB-XL Splits (Already Processed by NB1/NB2)

In [ ]:
ptbxl_train_path = SAVE_DIR / "train_processed.npz"
ptbxl_val_path   = SAVE_DIR / "val_processed.npz"

if not ptbxl_train_path.exists() or not ptbxl_val_path.exists():
    raise FileNotFoundError(
        "PTB-XL processed splits not found. Run NB1 (Dataset Loading) and "
        "NB2 (Preprocessing) first to generate train_processed.npz / "
        "val_processed.npz."
    )

d_train = np.load(ptbxl_train_path)
X_ptb_train = d_train["X"].astype(np.float32)   # (N, 1000, 12)
y_ptb_train = d_train["y"].astype(np.int64)
M_ptb_train = d_train["meta"].astype(np.float32)

d_val = np.load(ptbxl_val_path)
X_ptb_val = d_val["X"].astype(np.float32)
y_ptb_val = d_val["y"].astype(np.int64)
M_ptb_val = d_val["meta"].astype(np.float32)

print(f"PTB-XL train : X={X_ptb_train.shape}  y={y_ptb_train.shape}  meta={M_ptb_train.shape}")
print(f"PTB-XL val   : X={X_ptb_val.shape}  y={y_ptb_val.shape}  meta={M_ptb_val.shape}")

print("\nPTB-XL train class distribution:")
cnt = Counter(y_ptb_train)
for k in range(4):
    n = cnt.get(k, 0)
    print(f"  {RISK_LABELS[k]:10s}: {n:6,}  ({100*n/max(1,len(y_ptb_train)):.1f}%)")

Step 2: Configure INCART Path (Same Auto-Detect Logic as NB6)

In [ ]:
# ─── CONFIGURE INCART FOLDER (robust auto-detect with parent search) ────────
# Download from: https://physionet.org/content/incartdb/
# You can override by setting INCART_ROOT to a specific path string.

INCART_ROOT = None

candidates = []
cwd = Path.cwd()
ancestors = [cwd] + list(cwd.parents[:6])

for root in ancestors:
    candidates.extend([
        root / "data" / "incart" / "Training",
        root / "ECG_Project" / "data" / "incart" / "Training",
        root / "ECG_Project" / "data" / "incart",
        root / "data" / "incart",
    ])

try:
    sd = SAVE_DIR
    candidates.extend([
        sd / "incart" / "Training",
        sd / "incart",
        sd.parent / "incart" / "Training",
    ])
except Exception:
    pass

seen = set()
uniq_candidates = []
for p in candidates:
    if p is None:
        continue
    p = Path(p)
    if str(p) in seen:
        continue
    seen.add(str(p))
    uniq_candidates.append(p)

incart_path = None
for p in uniq_candidates:
    if p.exists() and any(p.glob("*.hea")):
        incart_path = p
        break

if incart_path is None:
    hea_files = list(Path(".").rglob("*.hea")) + list(Path(".").rglob("*.hea.gz"))
    incart_hea = [f for f in hea_files if "incart" in str(f).lower()]
    search_set = incart_hea if incart_hea else hea_files
    if len(search_set) > 0:
        parents = [f.parent for f in search_set]
        best_parent = max(set(parents), key=lambda d: parents.count(d))
        incart_path = best_parent

if INCART_ROOT:
    incart_path = Path(INCART_ROOT)

if incart_path is None or not incart_path.exists():
    raise FileNotFoundError(
        "INCART folder not found. Set INCART_ROOT to the correct folder or "
        "place .hea files under a data/incart/Training path."
    )

hea_list = sorted(list(incart_path.glob("*.hea")) + list(incart_path.glob("*.hea.gz")))
ALL_INCART_RECORDS = [p.stem for p in hea_list]

if len(ALL_INCART_RECORDS) == 0:
    raise FileNotFoundError(f"No .hea files found in {incart_path}")

print(f"Using INCART path: {incart_path.resolve()}")
print(f"Total INCART records found: {len(ALL_INCART_RECORDS)}")
print(f"First 5: {ALL_INCART_RECORDS[:5]}")

Step 3: Label Mapping & Signal Extraction (Identical Logic to NB6)

INCART uses the same AAMI beat annotation symbols as MIT-BIH. Max-risk
strategy: the highest-risk beat in each 10-second window determines the
label.

In [ ]:
# AAMI symbol → risk (same mapping as NB1 / MITBIH_TO_RISK)
SYMBOL_TO_AAMI = {
    "N": "N", ".": "N", "e": "N", "j": "N", "L": "N", "R": "N",
    "A": "S", "a": "S", "J": "S", "S": "S",
    "V": "V", "E": "V",
    "F": "F",
    "P": "N",
    "/": "Q", "Q": "Q", "?": "Q", "[": "Q", "]": "Q", "!": "Q", "f": "Q",
}
AAMI_TO_RISK = {"N": 0, "S": 1, "V": 2, "F": 2, "Q": 3}

unmapped_symbols = Counter()


def _normalize_symbol(sym):
    """Normalize annotation symbol to a printable single-char string."""
    if sym is None:
        return ""
    if isinstance(sym, (bytes, bytearray)):
        try:
            s = sym.decode("utf-8")
        except Exception:
            s = str(sym)
    else:
        s = str(sym)
    s = s.strip()
    if s == "":
        return s
    if s in SYMBOL_TO_AAMI:
        return s
    s_up = s.upper()
    if s_up in SYMBOL_TO_AAMI:
        return s_up
    if s[0] in SYMBOL_TO_AAMI:
        return s[0]
    if s[0].upper() in SYMBOL_TO_AAMI:
        return s[0].upper()
    return s


def reorder_leads(signal, sig_names, target_order=STANDARD_LEAD_ORDER):
    """Reorder signal columns to match standard clinical 12-lead order.
    Raises ValueError if a required lead is missing."""
    name_to_idx = {}
    for i, name in enumerate(sig_names):
        if isinstance(name, (bytes, bytearray)):
            try:
                n = name.decode("utf-8").upper()
            except Exception:
                n = str(name).upper()
        else:
            n = str(name).upper()
        name_to_idx[n] = i

    missing = [lead for lead in target_order if lead.upper() not in name_to_idx]
    if missing:
        raise ValueError(f"Missing leads {missing} in record leads: {sig_names}")

    idx_order = [name_to_idx[lead.upper()] for lead in target_order]
    return signal[:, idx_order]


def get_incart_segments(record_path):
    """Load INCART record → extract 10-second 12-lead segments → assign
    risk label. Returns list of (signal_1000x12, risk_label, meta_3)."""
    try:
        rec = wfdb.rdrecord(str(record_path))
        ann = wfdb.rdann(str(record_path), "atr")
    except Exception as e:
        print(f"  Skipping {record_path.name}: {e}")
        return []

    fs = getattr(rec, "fs", None)
    if fs is None:
        print(f"  Warning: sampling rate not found in {record_path.name}; skipping")
        return []

    try:
        signal = reorder_leads(rec.p_signal, rec.sig_name)   # (N_samples, 12)
    except ValueError as e:
        print(f"  Lead reorder failed for {record_path.name}: {e}")
        return []

    n_samp = signal.shape[0]
    seg_len_orig = int(10 * fs)   # 10-second window at native rate
    results = []

    ann_samples = list(getattr(ann, "sample", []))
    ann_symbols = list(getattr(ann, "symbol", []))

    for start in range(0, max(1, n_samp - seg_len_orig), seg_len_orig):
        end = start + seg_len_orig
        if end > n_samp:
            break
        seg = signal[start:end, :]   # (seg_len_orig, 12)

        ann_in_window = []
        for i, sidx in enumerate(ann_samples):
            if start <= sidx < end:
                sym_raw = ann_symbols[i] if i < len(ann_symbols) else ""
                sym = _normalize_symbol(sym_raw)
                ann_in_window.append(sym)

        if not ann_in_window:
            continue

        max_risk = 0
        for sym in ann_in_window:
            if sym not in SYMBOL_TO_AAMI and sym.upper() not in SYMBOL_TO_AAMI:
                unmapped_symbols[sym] += 1
            aami = SYMBOL_TO_AAMI.get(sym, SYMBOL_TO_AAMI.get(sym.upper(), "N"))
            risk = AAMI_TO_RISK.get(aami, 0)
            if risk > max_risk:
                max_risk = risk

        try:
            seg_resampled = scipy_resample(seg, TARGET_LEN, axis=0)   # (1000, 12)
        except Exception as e:
            print(f"  Resample failed for {record_path.name} at window {start}: {e}")
            continue

        mean = np.mean(seg_resampled, axis=0, keepdims=True)
        std = np.std(seg_resampled, axis=0, keepdims=True)
        std = np.where(std < 1e-8, 1e-8, std)
        seg_norm = (seg_resampled - mean) / std
        seg_norm = np.nan_to_num(seg_norm, nan=0.0, posinf=0.0, neginf=0.0)

        # Metadata defaults (INCART does not provide patient demographics)
        meta = np.array([0.6, 0.5, 0.375], dtype=np.float32)  # age~60/100, sex~0.5, hr~75/200

        results.append((seg_norm.astype(np.float32), max_risk, meta))

    return results

Step 4: Record-Level Train/Val Split for INCART

Splitting by **record** (not by segment) avoids leaking segments from the
same patient recording across train and val.

In [ ]:
rng = np.random.RandomState(SEED)
shuffled_records = list(ALL_INCART_RECORDS)
rng.shuffle(shuffled_records)

n_val_records = max(1, int(0.2 * len(shuffled_records)))
incart_val_records = set(shuffled_records[:n_val_records])
incart_train_records = set(shuffled_records[n_val_records:])

print(f"INCART records — train: {len(incart_train_records)}  val: {len(incart_val_records)}")

print("\nLoading INCART records...")
incart_train_segments, incart_val_segments = [], []

for rec_name in ALL_INCART_RECORDS:
    segs = get_incart_segments(incart_path / rec_name)
    if rec_name in incart_val_records:
        incart_val_segments.extend(segs)
    else:
        incart_train_segments.extend(segs)

if unmapped_symbols:
    print(f"⚠️  Unmapped symbols (defaulted to Low/Normal): {dict(unmapped_symbols)}")

if len(incart_train_segments) == 0 or len(incart_val_segments) == 0:
    raise RuntimeError("No segments extracted for INCART train or val split — check data.")


def stack_segments(segments):
    X = np.stack([s[0] for s in segments]).astype(np.float32)   # (N, 1000, 12)
    y = np.array([s[1] for s in segments], dtype=np.int64)
    M = np.stack([s[2] for s in segments]).astype(np.float32)   # (N, 3)
    return X, y, M


X_incart_train, y_incart_train, M_incart_train = stack_segments(incart_train_segments)
X_incart_val, y_incart_val, M_incart_val = stack_segments(incart_val_segments)

print(f"\nINCART train : X={X_incart_train.shape}  y={y_incart_train.shape}")
print(f"INCART val   : X={X_incart_val.shape}  y={y_incart_val.shape}")

print("\nINCART train class distribution:")
cnt = Counter(y_incart_train)
for k in range(4):
    n = cnt.get(k, 0)
    print(f"  {RISK_LABELS[k]:10s}: {n:6,}  ({100*n/max(1,len(y_incart_train)):.1f}%)")

Step 5: Assemble Joint Training & Validation Sets

PTB-XL-train + INCART-train → joint train.
PTB-XL-val + INCART-val → joint val (used for early stopping / checkpoint
selection). Per-source breakdowns are kept so Step 8 can report metrics
**separately** for each source as well as pooled.

In [ ]:
X_joint_train = np.concatenate([X_ptb_train, X_incart_train], axis=0)
y_joint_train = np.concatenate([y_ptb_train, y_incart_train], axis=0)
M_joint_train = np.concatenate([M_ptb_train, M_incart_train], axis=0)

X_joint_val = np.concatenate([X_ptb_val, X_incart_val], axis=0)
y_joint_val = np.concatenate([y_ptb_val, y_incart_val], axis=0)
M_joint_val = np.concatenate([M_ptb_val, M_incart_val], axis=0)

# Track source provenance for the joint val set so we can split results
# back out per dataset in Step 8.
source_joint_val = np.array(
    ["ptbxl"] * len(y_ptb_val) + ["incart"] * len(y_incart_val)
)

perm = np.random.RandomState(SEED).permutation(len(y_joint_train))
X_joint_train = X_joint_train[perm]
y_joint_train = y_joint_train[perm]
M_joint_train = M_joint_train[perm]

print(f"Joint train : X={X_joint_train.shape}  y={y_joint_train.shape}")
print(f"Joint val   : X={X_joint_val.shape}  y={y_joint_val.shape}")

print("\nJoint train class distribution:")
cnt = Counter(y_joint_train)
for k in range(4):
    n = cnt.get(k, 0)
    print(f"  {RISK_LABELS[k]:10s}: {n:6,}  ({100*n/max(1,len(y_joint_train)):.1f}%)")

del incart_train_segments, incart_val_segments
gc.collect()
print("\n✅ Joint dataset assembled.")

Step 6: Dataset / DataLoader (Same Recipe as NB4)

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, TensorDataset


class ECGDataset(Dataset):
    """Loads (N,1000,12) arrays → returns (12,1000) tensors + meta."""

    def __init__(self, X, y, meta, augment=False):
        self.X = X.astype(np.float32)
        self.y = y.astype(np.int64)
        self.meta = meta.astype(np.float32)
        self.augment = augment

    def _augment(self, x):
        x = x.copy()
        if np.random.rand() < 0.3:
            x += np.random.normal(0, 0.005, x.shape).astype(np.float32)
        if np.random.rand() < 0.3:
            x *= np.random.uniform(0.95, 1.05)
        if np.random.rand() < 0.3:
            x = np.roll(x, np.random.randint(-10, 10), axis=0)
        if np.random.rand() < 0.05:
            non_critical = [1, 2, 6, 7]
            x[:, np.random.choice(non_critical)] = 0.0
        mean = x.mean(axis=0, keepdims=True)
        std = np.where(x.std(axis=0, keepdims=True) < 1e-8, 1e-8, x.std(axis=0, keepdims=True))
        x = (x - mean) / std
        return np.nan_to_num(x, nan=0.0).astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].copy()
        if self.augment:
            x = self._augment(x)
        return (
            torch.from_numpy(x.T),                       # (12, 1000)
            torch.from_numpy(self.meta[idx]),             # (3,)
            torch.tensor(self.y[idx], dtype=torch.long),
        )


train_ds = ECGDataset(X_joint_train, y_joint_train, M_joint_train, augment=True)
val_ds   = ECGDataset(X_joint_val, y_joint_val, M_joint_val, augment=False)

cnt_train = Counter(y_joint_train)
class_sample_count = np.array([cnt_train.get(k, 1) for k in range(4)], dtype=np.float64)
weight_per_class = 1.0 / class_sample_count
sample_weights = np.array([weight_per_class[y] for y in y_joint_train])
sampler = WeightedRandomSampler(
    torch.from_numpy(sample_weights).double(), num_samples=len(sample_weights), replacement=True
)

BATCH_SIZE = 64
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=0, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True, drop_last=False,
)

print(f"✅ DataLoaders ready. Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

Step 7: Instantiate Model & Losses

Trains a **fresh** `ECGRiskNetXAI` from scratch on the joint PTB-XL +
INCART data (does not start from NB4's PTB-XL-only checkpoint, so the
comparison in Step 8 reflects what joint training alone achieves).

In [ ]:
model = ECGRiskNetXAI(in_ch=12, base_ch=64, meta_dim=3, dropout=0.3).to(device)

cnt_train = Counter(y_joint_train)
samples_per_class = [cnt_train.get(k, 1) for k in range(4)]
print(f"Samples per class (joint train): {samples_per_class}")

cb_focal = ClassBalancedFocalLoss(
    samples_per_class=samples_per_class, gamma=2.0, beta=0.9999, label_smoothing=0.1
).to(device)

supcon = SupConLoss(temperature=0.07)
aux_focal = FocalLoss(gamma=2.0, label_smoothing=0.1).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

total = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total:,}  ({total*4/1e6:.2f} MB)")


def mixup_batch(ecg, meta, labels, alpha=0.15):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    lam = max(lam, 1.0 - lam)
    B = ecg.size(0)
    idx = torch.randperm(B, device=ecg.device)
    mixed_ecg = lam * ecg + (1 - lam) * ecg[idx]
    mixed_meta = lam * meta + (1 - lam) * meta[idx]
    return mixed_ecg, mixed_meta, labels, labels[idx], lam


def mixup_loss(criterion, out_risk, la, lb, lam):
    return lam * criterion(out_risk, la) + (1 - lam) * criterion(out_risk, lb)


print("✅ Model, losses, and MixUp utility ready.")

Step 8: Training Loop

Same recipe as NB4 (CB-Focal + SupCon + auxiliary-head Focal, MixUp,
cosine warmup, SWA, gradient clipping, early stopping on val macro-F1).
Checkpoint is saved as `joint_model.pt` — NB4's `best_model.pt` is left
untouched.

In [ ]:
from torch.optim.swa_utils import AveragedModel, SWALR
from torch.optim.lr_scheduler import LambdaLR

NUM_EPOCHS    = 60
PATIENCE      = 15
SWA_START     = 52
MIXUP_ALPHA   = 0.15
SUPCON_W      = 0.1
AUX_W         = 0.15
GRAD_CLIP     = 1.0
WARMUP_EPOCHS = 5
BASE_LR       = 3e-4
MIN_LR        = 1e-6

swa_model = AveragedModel(model)
swa_sched = SWALR(optimizer, swa_lr=1e-5)


def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps,
                                     num_cycles=0.5, last_epoch=-1):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(
            max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + np.cos(np.pi * float(num_cycles) * 2.0 * progress)))
    return LambdaLR(optimizer, lr_lambda, last_epoch)


estimated_steps_per_epoch = len(train_loader)
warmup_scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_EPOCHS * estimated_steps_per_epoch,
    num_training_steps=NUM_EPOCHS * estimated_steps_per_epoch,
)

best_f1, best_epoch, patience_counter = 0.0, 0, 0
best_train_acc = 0.0
history = {
    "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
    "train_f1": [], "val_f1": [], "lr": [], "train_acc_clean": [],
}

step_count = 0

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # ─── TRAIN ───────────────────────────────────────────────────────────
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    tr_correct_clean, tr_total_clean = 0, 0
    tr_preds_all, tr_labels_all = [], []
    tr_preds_clean, tr_labels_clean = [], []

    for ecg_b, meta_b, y_b in train_loader:
        ecg_b = ecg_b.to(device)
        meta_b = meta_b.to(device)
        y_b = y_b.to(device)

        ecg_m, meta_m, la, lb, lam = mixup_batch(ecg_b, meta_b, y_b, MIXUP_ALPHA)

        optimizer.zero_grad()
        out = model(ecg_m, meta_m)

        risk_loss = mixup_loss(cb_focal, out["risk"], la, lb, lam)
        sc_loss = supcon(out["projection"], la)

        aux_loss = (
            aux_focal(out["arrhythmia"][:, :min(6, out["arrhythmia"].size(1))],
                       torch.clamp(la, 0, out["arrhythmia"].size(1) - 1))
            + aux_focal(out["mi"][:, :2], (la >= 3).long())
            + aux_focal(out["st_t"][:, :3], torch.clamp(la, 0, 2))
            + aux_focal(out["conduction"][:, :4], torch.clamp(la, 0, 3))
        ) / 4.0

        loss = risk_loss + SUPCON_W * sc_loss + AUX_W * aux_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        warmup_scheduler.step()
        step_count += 1

        tr_loss += loss.item()

        preds = out["risk"].argmax(1).detach().cpu()
        tr_correct += (preds == la.cpu()).sum().item()
        tr_total += len(la)
        tr_preds_all.extend(preds.numpy())
        tr_labels_all.extend(la.cpu().numpy())

        preds_clean = out["risk"].argmax(1).detach().cpu()
        tr_correct_clean += (preds_clean == y_b.cpu()).sum().item()
        tr_total_clean += len(y_b)
        tr_preds_clean.extend(preds_clean.numpy())
        tr_labels_clean.extend(y_b.cpu().numpy())

    tr_loss /= len(train_loader)
    tr_acc = tr_correct / tr_total
    tr_acc_clean = tr_correct_clean / tr_total_clean
    tr_f1 = f1_score(tr_labels_clean, tr_preds_clean, average="macro", zero_division=0)

    # ─── VALIDATE ────────────────────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    val_preds_all, val_labels_all = [], []

    with torch.no_grad():
        for ecg_b, meta_b, y_b in val_loader:
            ecg_b = ecg_b.to(device)
            meta_b = meta_b.to(device)
            y_b = y_b.to(device)
            out = model(ecg_b, meta_b)
            vloss = cb_focal(out["risk"], y_b)
            val_loss += vloss.item()
            preds = out["risk"].argmax(1).cpu()
            val_correct += (preds == y_b.cpu()).sum().item()
            val_total += len(y_b)
            val_preds_all.extend(preds.numpy())
            val_labels_all.extend(y_b.cpu().numpy())

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    val_f1 = f1_score(val_labels_all, val_preds_all, average="macro", zero_division=0)

    # ─── SWA update ──────────────────────────────────────────────────────
    if epoch >= SWA_START:
        swa_model.update_parameters(model)
        swa_sched.step()

    current_lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(tr_acc)
    history["train_acc_clean"].append(tr_acc_clean)
    history["val_acc"].append(val_acc)
    history["train_f1"].append(tr_f1)
    history["val_f1"].append(val_f1)
    history["lr"].append(current_lr)

    # ─── Early stopping & checkpoint (based on validation F1) ───────────
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        best_train_acc = tr_acc_clean
        patience_counter = 0
        torch.save(model.state_dict(), SAVE_DIR / "joint_model.pt")
    else:
        patience_counter += 1

    elapsed = time.time() - t0
    print(f"Ep {epoch:3d}/{NUM_EPOCHS} | "
          f"TrLoss {tr_loss:.4f} TrAcc {tr_acc_clean:.3f} TrF1 {tr_f1:.3f} | "
          f"VlLoss {val_loss:.4f} VlAcc {val_acc:.3f} VlF1 {val_f1:.3f} | "
          f"LR {current_lr:.2e} | {elapsed:.1f}s"
          + (" ★" if val_f1 == best_f1 else ""))

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (best val_f1={best_f1:.4f} at epoch {best_epoch})")
        break

# Update BN stats for SWA
try:
    torch.optim.swa_utils.update_bn(train_loader, swa_model)
    torch.save(swa_model.state_dict(), SAVE_DIR / "joint_swa_model.pt")
    print("✅ Joint SWA model saved.")
except Exception as e:
    print(f"SWA update_bn skipped: {e}")

print(f"\nBest val Macro F1 : {best_f1:.4f} (epoch {best_epoch})")
print(f"Train Accuracy at best epoch : {best_train_acc:.3f}")

Step 9: Training Curves

In [ ]:
epochs_ran = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].plot(epochs_ran, history["train_loss"], label="Train Loss", color="steelblue")
axes[0].plot(epochs_ran, history["val_loss"], label="Val Loss", color="orange")
axes[0].set_title("Loss — Joint PTB-XL + INCART Training", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, history["train_acc_clean"], label="Train Acc", color="steelblue")
axes[1].plot(epochs_ran, history["val_acc"], label="Val Acc", color="orange")
axes[1].set_title("Accuracy — Joint PTB-XL + INCART Training", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_ran, history["train_f1"], label="Train F1", color="steelblue")
axes[2].plot(epochs_ran, history["val_f1"], label="Val F1", color="orange")
axes[2].axvline(best_epoch, color="green", linestyle="--", alpha=0.6, label=f"Best (ep {best_epoch})")
axes[2].set_title("Macro F1 — Joint PTB-XL + INCART Training", fontweight="bold")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Macro F1")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_DIR / "joint_training_curves.png", dpi=150)
plt.show()

print("✅ Training curves saved.")

Step 10: Load Best Joint Checkpoint

In [ ]:
model.load_state_dict(torch.load(SAVE_DIR / "joint_model.pt", map_location=device))
model.eval()
print("✅ Best joint checkpoint reloaded for evaluation.")

Step 11: Separate Evaluation — PTB-XL-val, INCART-val, and Pooled

Per the spec, results are reported **separately** for each source dataset
(not just pooled), so we can see whether joint training helped or hurt
either dataset individually.

In [ ]:
def evaluate_array(X, y, M, split_name, batch_size=64):
    """Run the trained model on raw (X, y, meta) arrays already in
    (N, 1000, 12) format. Returns dict with all metrics + arrays."""
    X_t = torch.from_numpy(X.transpose(0, 2, 1))   # (N, 12, 1000)
    M_t = torch.from_numpy(M)
    y_t = torch.from_numpy(y)

    loader = DataLoader(TensorDataset(X_t, M_t, y_t), batch_size=batch_size,
                         shuffle=False, num_workers=0, pin_memory=True)

    all_preds, all_probs, all_true = [], [], []
    model.eval()
    with torch.no_grad():
        for xb, mb, yb in loader:
            out = model(xb.to(device), mb.to(device))
            probs = torch.softmax(out["risk"], dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
            all_preds.extend(preds)
            all_probs.append(probs)
            all_true.extend(yb.numpy())

    preds = np.array(all_preds)
    true = np.array(all_true)
    probs = np.vstack(all_probs)

    acc = (preds == true).mean()
    mf1 = f1_score(true, preds, average="macro", zero_division=0)
    wf1 = f1_score(true, preds, average="weighted", zero_division=0)
    mcc = matthews_corrcoef(true, preds)
    kappa = cohen_kappa_score(true, preds)

    per_class_f1 = f1_score(true, preds, average=None, zero_division=0, labels=[0, 1, 2, 3])
    per_class_prec = precision_score(true, preds, average=None, zero_division=0, labels=[0, 1, 2, 3])
    per_class_rec = recall_score(true, preds, average=None, zero_division=0, labels=[0, 1, 2, 3])

    roc_auc, pr_auc = 0.0, 0.0
    try:
        present = sorted(np.unique(true))
        if len(present) > 1:
            y_bin = label_binarize(true, classes=[0, 1, 2, 3])
            roc_auc = roc_auc_score(y_bin, probs, average="macro", multi_class="ovr", labels=[0, 1, 2, 3])
            pr_auc = np.mean([average_precision_score(y_bin[:, k], probs[:, k]) for k in range(4)])
    except Exception:
        pass

    print(f"\n{'='*55}")
    print(f"  {split_name} Results")
    print(f"{'='*55}")
    print(f"  Accuracy    : {acc*100:.2f}%")
    print(f"  Macro F1    : {mf1:.4f}")
    print(f"  Weighted F1 : {wf1:.4f}")
    print(f"  MCC         : {mcc:.4f}")
    print(f"  Cohen Kappa : {kappa:.4f}")
    print(f"  ROC-AUC     : {roc_auc:.4f}")
    print(f"  PR-AUC      : {pr_auc:.4f}")
    print()
    print(classification_report(true, preds, target_names=["Low", "Moderate", "High", "Critical"],
                                 digits=4, zero_division=0))

    return {
        "split": split_name, "accuracy": float(acc), "macro_f1": float(mf1),
        "weighted_f1": float(wf1), "mcc": float(mcc), "kappa": float(kappa),
        "roc_auc": float(roc_auc), "pr_auc": float(pr_auc),
        "per_class_f1": per_class_f1.tolist(), "per_class_prec": per_class_prec.tolist(),
        "per_class_rec": per_class_rec.tolist(),
        "_preds": preds, "_true": true, "_probs": probs,
    }


results_ptbxl_val = evaluate_array(X_ptb_val, y_ptb_val, M_ptb_val, "PTB-XL Val (joint-trained model)")
results_incart_val = evaluate_array(X_incart_val, y_incart_val, M_incart_val, "INCART Val (joint-trained model)")
results_joint_val = evaluate_array(X_joint_val, y_joint_val, M_joint_val, "Pooled Joint Val (PTB-XL + INCART)")

Step 12: Per-Class Metrics — Side by Side

In [ ]:
split_results = [
    (results_ptbxl_val, "PTB-XL Val", "steelblue"),
    (results_incart_val, "INCART Val", "green"),
    (results_joint_val, "Pooled Val", "purple"),
]

class_names = ["Low", "Moderate", "High", "Critical"]
x = np.arange(4)
width = 0.25

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
metrics = [
    ("per_class_f1", "Per-Class F1 Score"),
    ("per_class_prec", "Per-Class Precision"),
    ("per_class_rec", "Per-Class Recall"),
]

for ax, (key, title) in zip(axes, metrics):
    for i, (res, sname, color) in enumerate(split_results):
        vals = res[key]
        bars = ax.bar(x + i * width, vals, width, label=sname, color=color, alpha=0.8)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f"{val:.2f}", ha="center", fontsize=7)
    ax.set_xticks(x + width)
    ax.set_xticklabels(class_names)
    ax.set_title(title, fontweight="bold")
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle("Per-Class Metrics — Joint Model, by Source Dataset", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(SAVE_DIR / "joint_perclass_metrics_by_source.png", dpi=150)
plt.show()

print("✅ Per-class metrics plot saved.")

Step 13: Confusion Matrices — Side by Side

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
cmaps = ["Blues", "Greens", "Purples"]

for ax, (res, sname, _), cmap in zip(axes, split_results, cmaps):
    cm = confusion_matrix(res["_true"], res["_preds"], labels=[0, 1, 2, 3])
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix — {sname}  (Acc={res['accuracy']*100:.1f}%)", fontweight="bold")

plt.tight_layout()
plt.savefig(SAVE_DIR / "joint_confusion_matrices_by_source.png", dpi=150)
plt.show()

print("✅ Confusion matrices saved.")

Step 14: ROC & PR Curves — Side by Side

In [ ]:
cls_colors = ["#2ecc71", "#3498db", "#e67e22", "#e74c3c"]
split_styles = ["-", "--", ":"]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for (res, sname, _), linestyle in zip(split_results, split_styles):
    y_bin = label_binarize(res["_true"], classes=[0, 1, 2, 3])
    for k, color in zip(range(4), cls_colors):
        try:
            fpr, tpr, _ = roc_curve(y_bin[:, k], res["_probs"][:, k])
            auc_k = roc_auc_score(y_bin[:, k], res["_probs"][:, k])
            axes[0].plot(fpr, tpr, color=color, linestyle=linestyle, alpha=0.85,
                         label=f"{sname} — {RISK_LABELS[k]} (AUC={auc_k:.3f})")
        except Exception:
            pass

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_title("ROC Curves — Joint Model, by Source", fontweight="bold")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].legend(fontsize=7, ncol=2, loc="lower right")
axes[0].grid(True, alpha=0.3)

for (res, sname, _), linestyle in zip(split_results, split_styles):
    y_bin = label_binarize(res["_true"], classes=[0, 1, 2, 3])
    for k, color in zip(range(4), cls_colors):
        try:
            prec, rec, _ = precision_recall_curve(y_bin[:, k], res["_probs"][:, k])
            ap_k = average_precision_score(y_bin[:, k], res["_probs"][:, k])
            axes[1].plot(rec, prec, color=color, linestyle=linestyle, alpha=0.85,
                         label=f"{sname} — {RISK_LABELS[k]} (AP={ap_k:.3f})")
        except Exception:
            pass

axes[1].set_title("PR Curves — Joint Model, by Source", fontweight="bold")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=7, ncol=2, loc="lower left")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_DIR / "joint_roc_pr_by_source.png", dpi=150)
plt.show()

print("✅ ROC/PR curves saved.")

Step 15: Save Joint Training Results

In [ ]:
joint_training_results = {
    "ptbxl_val": {k: v for k, v in results_ptbxl_val.items() if not k.startswith("_")},
    "incart_val": {k: v for k, v in results_incart_val.items() if not k.startswith("_")},
    "pooled_val": {k: v for k, v in results_joint_val.items() if not k.startswith("_")},
    "best_epoch": best_epoch,
    "best_val_macro_f1": float(best_f1),
    "train_samples": int(len(y_joint_train)),
    "val_samples": int(len(y_joint_val)),
    "ptbxl_train_samples": int(len(y_ptb_train)),
    "incart_train_samples": int(len(y_incart_train)),
}

with open(SAVE_DIR / "joint_training_results.json", "w") as f:
    json.dump(joint_training_results, f, indent=2)

print()
print("NOTEBOOK 6c COMPLETE ✅")
print("=" * 55)
print(f"  PTB-XL Val   : Acc={results_ptbxl_val['accuracy']*100:.2f}%  MacroF1={results_ptbxl_val['macro_f1']:.4f}")
print(f"  INCART Val   : Acc={results_incart_val['accuracy']*100:.2f}%  MacroF1={results_incart_val['macro_f1']:.4f}")
print(f"  Pooled Val   : Acc={results_joint_val['accuracy']*100:.2f}%  MacroF1={results_joint_val['macro_f1']:.4f}")
print()
print("Saved:")
print(f"  {SAVE_DIR}/joint_model.pt")
print(f"  {SAVE_DIR}/joint_swa_model.pt")
print(f"  {SAVE_DIR}/joint_training_results.json")
print(f"  {SAVE_DIR}/joint_training_curves.png")
print(f"  {SAVE_DIR}/joint_perclass_metrics_by_source.png")
print(f"  {SAVE_DIR}/joint_confusion_matrices_by_source.png")
print(f"  {SAVE_DIR}/joint_roc_pr_by_source.png")
print()
print("Next → Notebook 6d: Cross-Dataset Validation on CPSC 2018 & Chapman-Shaoxing")